In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import faulthandler, sys
faulthandler.enable(file=sys.stderr)
print("faulthandler enabled", flush=True)

In [ ]:
# pip install matplotlib scikit-learn torchdiffeq numpy==2.0.0 jupyterplot siren_pytorch pinnstorch lightning pandas seaborn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import wandb
import os
from torch import nn
import warnings
warnings.filterwarnings('ignore')
wandb.init(mode="offline")
# os.environ["WANDB_NOTEBOOK_NAME"] = "PINN_GPE.ipynb"
wandb.login(host='http://localhost:8080', 
            key='local-0f9731927571d8659bd45b0d14c25d90e730aa0f')
run = wandb.init(
    project="PINN_0",
    settings=wandb.Settings(
        base_url="http://localhost:8080"
    )
)

# run = wandb.init(mode="offline", project="PINN2")
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
np.random.seed(0xFA1AFE1)

data = np.load('data_demo/GPE_phase_PHASE_3D_10x10x10_R_7067c_BACKUP_PRESENT_0_intermediate_0.npz')

In [ ]:
np.random.seed(42)

In [ ]:
import matplotlib
matplotlib.rcdefaults() 
plt.style.use('meth.mplstyle')

In [ ]:
np.random.seed(42)

In [ ]:
try:
    if torch.cuda.is_available():
        device = 'cuda:0'
    else:
        device = 'cpu:0'
except:
    device = 'cpu:0'

In [ ]:
# device = torch.device("cpu")

In [ ]:
# from src.data.dataset import GPEDataset, create_dataloaders
# data_ic = GPEDataset('../Thesis/datasets/gpe_simulations/', normalize=False, mode='initial_conditions')
# data_traj = GPEDataset('../Thesis/datasets/gpe_simulations/', normalize=False, mode='trajectories')

In [ ]:
# train_loader_ic, val_loader_ic, test_loader_ic = create_dataloaders('../Thesis/datasets/gpe_simulations/', batch_size=2, normalize=False, mode='initial_conditions')
# train_loader_traj, val_loader_traj, test_loader_traj = create_dataloaders('../Thesis/datasets/gpe_simulations/', batch_size=2,  normalize=False, mode='trajectories')

In [ ]:
# beta = 10.
# step = 0.05
beta = 1.
step = 0.001
# V = 1000
# N = 10
N = 4
V = N ** 3
latent_dim = 128
N_steps = 5000
# N_steps = 20000

In [ ]:
# from src.models.vae import VAE
# vae = VAE(2*V, latent_dim=latent_dim)
# # vae_chkpt = torch.load('../Thesis/outputs/vae/checkpoints/best_checkpoint.pt')
# vae_chkpt = torch.load('outputs/outputs/vae/checkpoints/best_checkpoint.pt')
# vae.load_state_dict(vae_chkpt['model_state_dict'], strict=False)

In [ ]:
# # vae.encode(next(train_loader_ic))
# for item in train_loader_ic:
#     print(item[0].shape, item[1])
#     break
# for item in train_loader_traj:
#     print(item[0].shape, item[1])
#     break

In [ ]:
# data_traj[0][0].shape

In [ ]:
# len(data_traj)#[0]#[0][0]

In [ ]:
# X, Y = data_traj[0][0][0], data_traj[0][0][1]
# print(X.shape, Y.shape)
# e = 6
# print(data['order_parameters'][0,e,:,:,:,:].real.shape)

In [ ]:
from DGPE.GPElib.lyapunov_generator import LyapunovGenerator
from DGPE.GPElib.dynamics_generator import DynamicsGenerator

dgpe = DynamicsGenerator(N_part_per_well=1.,
                         W=0, disorder_seed=53,
                         N_wells=(N,N,N), dimensionality=3, anisotropy=1.,
                         threshold_XY_to_polar=0.25,
                         J=1, beta=beta,  #beta_disorder_amplitude=2.,
                        integration_method='RK45',
                         rtol=1e-8, atol=1e-8,
                        # smooth_quench=True,
                         smooth_quench_to_room=True,
                         reset_steps_duration=5,
                         calculation_type='lyap_save_all',
                        integrator='scipy',
                        # gpu_integrator='torch',
                        # gpu_id=0,
                         time=step*N_steps+1, step=step, t_steps=N_steps, gamma=1.,
                         quenching_gamma=1.)

e = 6
# dgpe.X, dgpe.Y = data['order_parameters'][0,e,:,:,:,:].real, data['order_parameters'][0,e,:,:,:,:].imag
dgpe.X.shape, dgpe.Y.shape
# dgpe.set_init_XY(data['order_parameters'][0,e,:,:,:,0].real, data['order_parameters'][0,e,:,:,:,0].imag)
traj_seed = 42
E = 1
dgpe.generate_init(traj_seed, E, kind='random_population_and_phase')
dgpe.set_init_XY(dgpe.X[:,:,:,0], dgpe.Y[:,:,:,0])
dgpe.step = step
dgpe.n_steps = N_steps
dgpe.icurr = 0
dgpe.inext = 1

dgpe.run_dynamics(no_pert=False)

In [ ]:
dgpe.X.shape
plt.plot(dgpe.X[0,0,0,:])
plt.show()

In [ ]:
dgpe.X.shape

In [ ]:
# j = 0
# dgpe.X = data_traj[j][0][0]
# dgpe.Y = data_traj[j][0][1]

In [ ]:
# wSIM ∝ 1,
# wIBC ∝ exp(− lSIM ,)
# wPDE ∝ exp(− max(lSIM, lIBC),)

In [ ]:
# j = 0
# dgpe.X = data_traj[j][0][0].numpy()
# dgpe.Y = data_traj[j][0][1].numpy()

In [ ]:
# X0, Y0 = data_traj[j][0][0][:,:,:,0], data_traj[j][0][1][:,:,:,0]
# print(X0.shape, Y0.shape)
# print(torch.stack((X0.flatten(),Y0.flatten())).flatten().shape)
# z0 = vae.encode(torch.stack((X0.flatten(),Y0.flatten())).flatten())
# z0 = torch.stack(z0).flatten()
# print(z0.shape)

In [ ]:
dgpe.X.shape

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
X = step * np.arange(dgpe.X.shape[-1]).reshape(-1,1)
# y = np.hstack((np.moveaxis(dgpe.X, -1, 0).reshape(-1, dgpe.N_wells), np.moveaxis(dgpe.Y, -1, 0).reshape(-1, dgpe.N_wells)))
y = np.stack((np.moveaxis(dgpe.X, -1, 0), np.moveaxis(dgpe.Y, -1, 0)), axis=1)
print(y.shape)
train_size = int(0.5 * X.shape[0])
X_train_raw = X[:train_size, :]
y_train_raw = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]
X_train_raw.shape, X_test.shape

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.5, random_state=0xE2E4
)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
from src.dgpe_nn import DGPEModule, UNetDGPEModule
from src.pinn_lib import train_and_validate, plot_losses, UNet_train_and_validate
from src.dataloaders import generate_datasets, UNet_generate_datasets

In [ ]:
batch_size = 512
# istride = 500
# istride = 100
# istride = 150
# istride = 50
istride = 5

# train_loader, test_loader, val_loader, init_loader = generate_datasets(X, y, X_train, y_train, X_test, y_test, X_val, y_val, batch_size, istride)
train_loader, test_loader, val_loader, init_loader = UNet_generate_datasets(X, y, X_train, y_train, X_test, y_test, X_val, y_val, batch_size, istride, step=step)

In [ ]:
from sklearn.preprocessing import LabelEncoder
n_hidden = 4

# n_in, n_out = X.shape[-1] + z0.shape[0], y.shape[-1]
n_in, n_out = X.shape[-1], y.shape[-1]
# model = DGPEModule(dgpe, n_in, n_out)

# max_norm = 100.0  # Define a threshold
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
# # optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.95, weight_decay=1e-3)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)#, momentum=0.95, weight_decay=1e-3)
# def crit(x,y):
#     print(x.shape, y.shape)
#     return nn.MSELoss()(x,y)
# # criterion = nn.MSELoss()
# criterion = crit
# metric = lambda x,y: nn.MSELoss()(x, y)
# num_epochs = 30

# wandb.watch(model, criterion, log="all", log_freq=10) # Log gradients and parameters

In [ ]:
from sklearn.preprocessing import LabelEncoder
# n_hidden = 128
# n_hidden = 32
n_hidden = 16
# n_hidden = 8
# n_hidden = 4

# n_in, n_out = X.shape[-1], y.shape[-1]
# real/imag — 2 channels, 4x4x4 grid
input_shape = (2,N,N,N)
model = UNetDGPEModule(dgpe, input_shape, n_hidden=n_hidden)
model2 = UNetDGPEModule(dgpe, input_shape, n_hidden=n_hidden)
model3 = UNetDGPEModule(dgpe, input_shape, n_hidden=n_hidden)

# model = DGPEModule(dgpe, n_in, n_out, n_hidden=n_hidden)
# model2 = DGPEModule(dgpe, n_in, n_out, n_hidden=n_hidden)
# max_norm = 100.0  # Define a threshold
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3,  weight_decay=1e-3)
#clip gradients
# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)

criterion = nn.MSELoss()
metric = lambda x,y: nn.MSELoss()(x, y)
num_epochs = 30
wandb.watch(model, criterion, log="all", log_freq=10) # Log gradients and parameters
wandb.watch(model2, criterion, log="all", log_freq=10) # Log gradients and parameters
wandb.watch(model3, criterion, log="all", log_freq=10) # Log gradients and parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

In [ ]:
# vae.encode(psi)
# vae.encoder.forward(psi)

In [ ]:
ne = 10
max_grad = 1e+3
# max_grad = 1e+2
N1, N2, N3 = 3, 30, 300
# N1, N2, N3 = 30, 300, 3000
vae = None
lr_small = 1e-5
lr_large = 1e-3
stages = [
    # (lr, epochs_mult, flags)
    (lr_small, N1, dict(criterion_init_cond=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True, criterion_pinn=True)),
    
    (lr_large, N2, dict(criterion_init_cond=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True)),
    (lr_large, N3, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True, criterion_pinn=True)),
]

for lr, emult, flags in stages:
    # optimizer = torch.optim.LBFGS(model.parameters(), lr=lr)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_epochs = emult * ne
    UNet_train_and_validate(
    # train_and_validate(
        model, optimizer, criterion, metric,
        train_loader, val_loader, init_loader,
        num_epochs,
        w1=1., w2=1., w3=1., w4=1e-3, w5=1e-6, w6=1., w7=1.,
       run=run, verbose=False,
       patience = 10,
       # patience = 1000,
       max_grad_norm=max_grad,
        vae=vae,
        **flags,
    )

In [ ]:
# ne = 10
# max_grad = 1e+3
# N1, N2, N3 = 3, 30, 300
# N1, N2, N3 = 30, 300, 3000
vae = None
stages = [
    # (lr, epochs_mult, flags)
    (lr_small, N1, dict(criterion_init_cond=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True, criterion_pinn=False)),
    
    (lr_large, N2, dict(criterion_init_cond=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True)),
    (lr_large, N3, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=True, criterion_Econst=True, criterion_pinn=False)),
]


for lr, emult, flags in stages:
    # optimizer = torch.optim.LBFGS(model2.parameters(), lr=lr)
    optimizer = torch.optim.Adam(model2.parameters(), lr=lr)
    num_epochs = emult * ne
    UNet_train_and_validate(
        model2, optimizer, criterion, metric,
        train_loader, val_loader, init_loader,
        num_epochs,
        w1=1., w2=1., w3=1., w4=1e-3, w5=1e-6, w6=1., w7=1.,
       run=run, verbose=False,
       patience = 10,
       # patience = 1000,
       max_grad_norm=max_grad,
        vae=vae,
        **flags,
    )

In [ ]:
# ne = 10
# max_grad = 1e+3
# N1, N2, N3 = 3, 30, 300
# N1, N2, N3 = 30, 300, 3000
vae = None
stages = [
    # (lr, epochs_mult, flags)
    (lr_small, N1, dict(criterion_init_cond=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False, criterion_Econst=False)),
    (lr_small, N1, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False, criterion_Econst=False, criterion_pinn=False)),
    
    (lr_large, N2, dict(criterion_init_cond=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False)),
    (lr_large, N2, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False, criterion_Econst=False)),
    (lr_large, N3, dict(criterion_init_cond=True, criterion_ibound=True, criterion_Nconst=False, criterion_Econst=False, criterion_pinn=False)),
]


for lr, emult, flags in stages:
    optimizer = torch.optim.Adam(model3.parameters(), lr=lr)
    # optimizer = torch.optim.LBFGS(model3.parameters(), lr=lr)
    num_epochs = emult * ne
    UNet_train_and_validate(
        model3, optimizer, criterion, metric,
        train_loader, val_loader, init_loader,
        num_epochs,
        w1=1., w2=1., w3=1., w4=1e-3, w5=1e-6, w6=1., w7=1.,
       run=run, verbose=False,
       patience = 10,
       # patience = 1000,
       max_grad_norm=max_grad,
        vae=vae,
        **flags,
    )

In [ ]:
# print(z0.shape, t.shape)
# torch.cat((t, z0)).shape

In [ ]:
dgpe.X.shape, dgpe.Y.shape

In [ ]:
psi_exp = np.hstack((np.moveaxis(dgpe.X, -1,0).reshape(N_steps, -1), 
          np.moveaxis(dgpe.Y, -1,0).reshape(N_steps, -1)))
psi_exp.shape

In [ ]:
ts = torch.tensor(step*np.arange(N_steps), dtype=torch.float32, requires_grad=False).reshape(-1,1)
# z0s = z0.broadcast_to(N_steps, z0.shape[0])
# ts.shape, z0s.shape

In [ ]:
# input_params = torch.concat((ts, z0s), 1)
input_params = ts
input_params.shape

In [ ]:
# psi = model(torch.cat((t, z0)))
# psi = model(input_params)
def _t(a):
    return torch.as_tensor(a, dtype=torch.float32)
spatial = _t(y).shape[2:]
dt_map_train = _t(X[istride::istride] - X[:-istride:istride]).view(-1, 1, 1, 1, 1).expand(-1, 1, *spatial)
y_input = torch.cat([_t(y[:-istride:istride]), dt_map_train], dim=1)
# dt_map_train = _t(ts[istride::istride]).view(-1, 1, 1, 1, 1).expand(-1, 1, *spatial)
# y_input = torch.cat([_t(y[0]).unsqueeze(0).expand(dt_map_train.shape[0], -1, *spatial), dt_map_train], dim=1)
y_output = y[istride::istride]
psi = model(y_input)
psi = psi.detach().numpy()

In [ ]:
# psi2 = model2(input_params)
psi2 = model2(y_input)
psi2 = psi2.detach().numpy()

In [ ]:
# psi2 = model2(input_params)
psi3 = model3(y_input)
psi3 = psi3.detach().numpy()

In [ ]:
dt_map_train = _t(X[istride::istride] - X[:-istride:istride]).view(-1, 1, 1, 1, 1).expand(-1, 1, *spatial)
y_input = torch.cat([_t(y[:-istride:istride]), dt_map_train], dim=1)
print(y_input.shape, dt_map_train.shape)
for i in range(1,dt_map_train.shape[0]):
    # print(y_input[i-1].unsqueeze(0).shape)
    # print(model3(y_input[i-1].unsqueeze(0)).reshape(1, 2, N, N, N).shape)
    # print(dt_map_train[0].unsqueeze(0).shape)
    # print(model3(y_input[i-1].unsqueeze(0)).reshape(1, 2, N, N, N))
    # print(dt_map_train[0].unsqueeze(0))
    # print(torch.cat([model3(y_input[i-1].unsqueeze(0)).detach().reshape(1, 2, N, N, N), dt_map_train[0].unsqueeze(0)], dim=1).shape)
    y_input[i] = torch.cat([model3(y_input[i-1].unsqueeze(0)).detach().reshape(1, 2, N, N, N), dt_map_train[0].unsqueeze(0)], dim=1)

In [ ]:
psi4 = model3(y_input)
psi4 = psi4.detach().numpy()

In [ ]:
psi.shape, psi2.shape

In [ ]:
psi_exp.shape, psi_exp[istride::istride].shape

In [ ]:
# time.shape, psi.shape, psi2.shape, time[istride::istride].shape

In [ ]:
np.random.seed(42)
time = np.arange(N_steps) * step
for i_plot in np.random.randint(0, 2 * V, 3):
    plt.figure(figsize=(12,9), dpi=300)
    plt.plot(time, psi_exp[:,i_plot], color='blue', label='Exact simulation')
    plt.plot(time[istride::istride], psi[:,i_plot], color='r', label='Prediction NN w PINN & w IMs')
    plt.plot(time[istride::istride], psi2[:,i_plot], color='g', label='Prediction NN w/o PINN & w IMs')
    plt.plot(time[istride::istride], psi3[:,i_plot], color='orange', label='Prediction NN w/o PINN & w/o IMs')
    plt.scatter(time[::istride], psi_exp[::istride, i_plot], s=30, color='blue', label='Exact simulation')
    plt.axvline(0.5*time[-1], linestyle='dashed', color='gray', label='Interpolation/extrapolation split')
    plt.legend(fontsize=16)
    plt.xlabel('Time')
    if i_plot < V:
        plt.ylabel(rf'Real $\Psi_{{{i_plot}}}$')
    else:
        plt.ylabel(rf'Imag $\Psi_{{{i_plot % V}}}$')        
    plt.axhline(0, color='k')
    plt.axvline(0, color='k')
    plt.tight_layout()
    plt.savefig(f'pics/UNet_examples_{i_plot}.png', format='png')
    plt.show()

In [ ]:
time = np.arange(N_steps) * step
mse_err = np.mean((psi - psi_exp[istride::istride]) ** 2, axis=1)
mse_err2 = np.mean((psi2 - psi_exp[istride::istride]) ** 2, axis=1)
mse_err3 = np.mean((psi3 - psi_exp[istride::istride]) ** 2, axis=1)
plt.figure(figsize=(12,9), dpi=300)
# plt.plot(time, psi_exp[:,i_plot], color='blue', label='Exact simulation')
plt.plot(time[istride::istride], mse_err, color='r', label='Prediction NN w PINN & w IMs')
plt.plot(time[istride::istride], mse_err2, color='g', label='Prediction NN w/o PINN & w IMs')
plt.plot(time[istride::istride], mse_err3, color='orange', label='Prediction NN w/o PINN & w/o IMs')
# plt.scatter(time[::istride], psi_exp[::istride, i_plot], s=30, color='blue', label='Exact simulation')
plt.axvline(0.5*time[-1], linestyle='dashed', color='gray', label='Interpolation/extrapolation split')
plt.legend(fontsize=16)
plt.xlabel('Time')
plt.ylabel('MSE')
plt.axhline(0, color='k')
plt.axvline(0, color='k')
plt.axis('on')
plt.tight_layout()
plt.savefig(f'pics/UNet_MSE.png', format='png')
plt.show()

In [ ]:
# https://en.wikipedia.org/wiki/Kosambi%E2%80%93Karhunen%E2%80%93Lo%C3%A8ve_theorem

In [ ]:
# chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://ocw.mit.edu/courses/18-s096-matrix-calculus-for-machine-learning-and-beyond-january-iap-2023/mit18_s096iap23_lec09.pdf

In [ ]:
1